<a href="https://colab.research.google.com/github/emmanuelokereke20-ctrl/AI-BASED-PHISHING-EMAIL-SECURITY-SYSTEMS/blob/main/Copy_of_STAGE_3_Explained.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import files
import pandas as pd
import numpy as np

# This opens a file picker — browse to dataset_phishing.csv on your laptop
uploaded = files.upload()

df = pd.read_csv('dataset_phishing.csv')
print(df.shape)
print(df['status'].value_counts())

X = df.drop(['status', 'url'], axis=1, errors='ignore')
y = (df['status'] == 'phishing').astype(int)
print(f"Features: {X.shape[1]}, Samples: {len(y)}")

Saving dataset_phishing.csv to dataset_phishing.csv
(11430, 89)
status
legitimate    5715
phishing      5715
Name: count, dtype: int64
Features: 87, Samples: 11430


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# 80/20 stratified split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Scale features — fitted on training data only
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Non-IID partition into 5 Trust nodes
trust_sizes = [3657, 2286, 1828, 914, 459]
trust_labels = ['A', 'B', 'C', 'D', 'E']

trust_nodes = {}
start = 0
for label, size in zip(trust_labels, trust_sizes):
    trust_nodes[label] = {
        'X': X_train_scaled[start:start + size],
        'y': y_train.iloc[start:start + size].values
    }
    start += size
    print(f"Trust {label}: {size} samples")

print(f"\nTest set: {len(X_test_scaled)} samples")

Trust A: 3657 samples
Trust B: 2286 samples
Trust C: 1828 samples
Trust D: 914 samples
Trust E: 459 samples

Test set: 2286 samples


In [ ]:
from sklearn.neural_network import MLPClassifier
import copy

# Configure the federated model architecture
global_model = MLPClassifier(
    hidden_layer_sizes=(64, 32),
    activation='relu',
    warm_start=True,
    max_iter=10,
    random_state=42
)

# Pre-fit on 100 bootstrap samples from Trust A to establish
# internal weight matrix dimensions — no real data leaks in
bootstrap_idx = np.random.choice(len(trust_nodes['A']['X']), 100, replace=True)
global_model.fit(
    trust_nodes['A']['X'][bootstrap_idx],
    trust_nodes['A']['y'][bootstrap_idx]
)

print("Global model initialised")
print(f"Weight matrix shapes: {[c.shape for c in global_model.coefs_]}")

Global model initialised
Weight matrix shapes: [(87, 64), (64, 32), (32, 1)]


/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (10) reached and the optimization hasn't converged yet.
  warnings.warn(


In [ ]:
from sklearn.metrics import accuracy_score, f1_score
import warnings
warnings.filterwarnings('ignore')

n_rounds = 20
total_samples = sum(trust_sizes)
results = []

for rnd in range(1, n_rounds + 1):
    local_coefs = []
    local_intercepts = []

    # Each Trust node trains a local copy
    for label, size in zip(trust_labels, trust_sizes):
        local_model = copy.deepcopy(global_model)
        local_model.fit(
            trust_nodes[label]['X'],
            trust_nodes[label]['y']
        )
        local_coefs.append((local_model.coefs_, size))
        local_intercepts.append((local_model.intercepts_, size))

    # FedAvg aggregation — weighted average by node size
    new_coefs = []
    for layer_idx in range(len(global_model.coefs_)):
        layer_avg = sum(
            coefs[layer_idx] * (size / total_samples)
            for coefs, size in local_coefs
        )
        new_coefs.append(layer_avg)

    new_intercepts = []
    for layer_idx in range(len(global_model.intercepts_)):
        layer_avg = sum(
            intercepts[layer_idx] * (size / total_samples)
            for intercepts, size in local_intercepts
        )
        new_intercepts.append(layer_avg)

    # Update global model weights
    global_model.coefs_ = new_coefs
    global_model.intercepts_ = new_intercepts

    # Evaluate on held-out test set
    y_pred = global_model.predict(X_test_scaled)
    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    fpr = sum((y_pred == 1) & (y_test == 0)) / sum(y_test == 0)

    results.append({'round': rnd, 'accuracy': acc, 'f1': f1, 'fpr': fpr})
    print(f"Round {rnd:2d}/20  |  accuracy {acc*100:.2f}%  |  F1 {f1:.4f}  |  FPR {fpr*100:.2f}%")

Round  1/20  |  accuracy 94.05%  |  F1 0.9401  |  FPR 5.34%
Round  2/20  |  accuracy 94.66%  |  F1 0.9464  |  FPR 4.90%
Round  3/20  |  accuracy 94.75%  |  F1 0.9474  |  FPR 5.07%
Round  4/20  |  accuracy 94.93%  |  F1 0.9491  |  FPR 4.81%
Round  5/20  |  accuracy 95.06%  |  F1 0.9505  |  FPR 4.72%
Round  6/20  |  accuracy 95.01%  |  F1 0.9500  |  FPR 4.64%
Round  7/20  |  accuracy 95.14%  |  F1 0.9513  |  FPR 4.64%
Round  8/20  |  accuracy 95.28%  |  F1 0.9526  |  FPR 4.46%
Round  9/20  |  accuracy 95.32%  |  F1 0.9531  |  FPR 4.55%
Round 10/20  |  accuracy 95.32%  |  F1 0.9532  |  FPR 4.72%
Round 11/20  |  accuracy 95.23%  |  F1 0.9524  |  FPR 4.99%
Round 12/20  |  accuracy 95.28%  |  F1 0.9529  |  FPR 4.99%
Round 13/20  |  accuracy 95.10%  |  F1 0.9511  |  FPR 5.07%
Round 14/20  |  accuracy 95.10%  |  F1 0.9511  |  FPR 5.07%
Round 15/20  |  accuracy 95.14%  |  F1 0.9515  |  FPR 5.07%
Round 16/20  |  accuracy 95.19%  |  F1 0.9520  |  FPR 4.99%
Round 17/20  |  accuracy 95.28%  |  F1 0

In [ ]:
import pandas as pd

results_df = pd.DataFrame(results)
print("\n=== Final Result (Round 20) ===")
final = results_df.iloc[-1]
print(f"Accuracy : {final['accuracy']*100:.2f}%")
print(f"F1 Score : {final['f1']:.4f}")
print(f"FPR      : {final['fpr']*100:.2f}%")
print(f"\nCentralised baseline (Gradient Boosting): 95.36%")
print(f"Federated gap: {(0.9536 - final['accuracy'])*100:.2f} percentage points")


=== Final Result (Round 20) ===
Accuracy : 95.10%
F1 Score : 0.9512
FPR      : 5.25%

Centralised baseline (Gradient Boosting): 95.36%
Federated gap: 0.26 percentage points
